In [1]:
# "D:\WRI\Field Boundaries\NICFI zeroshot FTW scaled\Ucayali_NICFI_Mosaic_Jan-Jun_2021.tif"

In [1]:
import rasterio
import numpy as np

tif_path = r"D:\WRI\Field Boundaries\NICFI zeroshot FTW scaled\Ucayali_NICFI_Mosaic_Jan-Jun_2021.tif"

with rasterio.open(tif_path) as src:
    data = src.read()  # Reads all bands
    data = np.ma.masked_array(data, data == src.nodata)  # Mask nodata if defined

    min_val = data.min()
    max_val = data.max()

print(f"Value range: {min_val} to {max_val}")


Value range: 0.0 to 3.2073333263397217


Scale correctly; multiple by 10k and make into integer

In [1]:
import os
import rasterio
import numpy as np

input_dir = r"D:\WRI\Field Boundaries\NICFI zeroshot FTW scaled"
output_dir = os.path.join(input_dir, "truescale")
os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if filename.lower().endswith(".tif"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        with rasterio.open(input_path) as src:
            profile = src.profile.copy()
            data = src.read()

            nodata_float = src.nodata
            new_nodata = -9999

            # Replace original nodata with new integer nodata
            if nodata_float is not None:
                data = np.where(data == nodata_float, new_nodata, data * 10000)
            else:
                data = data * 10000

            scaled_data = data.astype(np.int32)

            profile.update(
                dtype=rasterio.int32,
                nodata=new_nodata
            )

            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(scaled_data)

        print(f"Processed: {filename} → truescale version saved.")

print("All files processed and saved to 'truescale'.")


C:\Users\grupp\AppData\Local\Temp\ipykernel_44756\883762950.py:23: RuntimeWarning: overflow encountered in multiply
  data = np.where(data == nodata_float, new_nodata, data * 10000)


RasterioIOError: Write failed. See previous exception for details.